# Defining Boundary Conditions

Written for dolfinx v0.7.2, python 3.10.6

With the mesh completed, the next step is to define the values of the fields at the boundary of the mesh.

# Libraries / Packages

In [14]:
from mpi4py import MPI # This is the communicator (or message passing interface) that allows python to work in parallel

# Create the MPI communicator (This has to happen before any other code is run.)
comm = MPI.COMM_WORLD
print(f"size = {comm.Get_size()}")
print(f"rank = {comm.Get_rank()}")

import numpy as np # For working with arrays
import dolfinx # Dolfinx is FEniCSx's computation backend / user interface
import leopart.cpp as pyleopart
import leopart.io
import json # For mapping entity names to physical groups
import ufl

size = 1
rank = 0


# Variables & Parameters

In [2]:
import variables

# Input Pre-Defined Mesh

This is collimator_tetra.msh file that was defined in gmsh_collimator.ipynb. GMSH is separate to FEniCSx, and FEniCSx requires the mesh to be in a xmdf format. The file is converted using meshio. 

In [3]:
# Read mesh from the XDMF file (it automatically reads the linked .h5 file)
with dolfinx.io.XDMFFile(comm, "meshes/collimator_tetra.xdmf", "r") as xdmf: 
    mesh = xdmf.read_mesh(name="Grid") 
    print("mesh read") 
    cell_tags = xdmf.read_meshtags(mesh, name="Grid") 
    print("cell tags read")
mesh.topology.create_connectivity(mesh.topology.dim, mesh.topology.dim - 1)
with dolfinx.io.XDMFFile(comm, "meshes/collimator_tetra_facets.xdmf", "r") as xdmf:
    facet_tags = xdmf.read_meshtags(mesh, name="Grid")
    print("facet tags read")

mesh read
cell tags read
facet tags read


# Read in Physical Group Tags

The physical groups are identified by a tag, not by name. To make the code more readable, we map these tags to names. 
To retrieve the tag of the anode, for example, use `physicalgroups['anode']['tag']`

In [4]:
with open("physicalgroups.json", "r") as file:
    physicalgroups = json.load(file)

print("Named physical groups:")
for group, properties in physicalgroups.items():
    print(f"{group}: dim = {properties['dim']}, tag = {properties['tag']}")

Named physical groups:
points: dim = 0, tag = 59
lines: dim = 1, tag = 60
collimator_surface: dim = 2, tag = 54
anode_surface: dim = 2, tag = 55
cathode_surface: dim = 2, tag = 56
trigger_surface: dim = 2, tag = 57
magnets_surface: dim = 2, tag = 58
collimator: dim = 3, tag = 1
anode: dim = 3, tag = 2
cathode: dim = 3, tag = 3
trigger: dim = 3, tag = 4
magnets: dim = 3, tag = 5


# Calculate static magnetic field due to magnets

First, the static magnetic field due to the magnets, cathode and anode must be determined. 

To do this, the variational form of Maxwell's equations must be determined. 

To determine the static B field, the two relevant equations are:

$$
\vec{\nabla} \cdot \vec{B} = 0 \tag{1}
$$

$$
\vec{\nabla} \times \vec{H} = \frac{\partial \vec{D}}{\partial t} + \vec{J} \tag{2}
$$

The static $\vec{B}$ field is time-invariant by definition($\therefore \frac{\partial \vec{D}}{\partial t} = 0$), and there are no moving charges (\therefore \vec{J} = 0$), so therefore:

$$
\vec{\nabla} \times \vec{H} = 0
$$

If the curl of a vector is zero, then it can be described as the gradient of a scalar potential. In the case of $\vec{H}$, this is the scalar magnetic potential $\phi$. 

$$
\vec{H} = - \vec{\nabla} \phi
$$

We also know that the magnetic field around a permanent magnet can be described by:

$$ 
\vec{B} = \mu_0 (\vec{H} + \vec{M})
$$

Where $\vec{H}$ is the magnetic field strength, and $\vec{M}$ is the magnetisation (or, density of magnetic dipole moments in a magnetic material).

$$
\therefore \vec{H} = \frac{1}{\mu_0} \vec{B} - \vec{M}
$$
$$
\therefore \vec{\nabla} \times \vec{H} = \vec{\nabla} \times (\frac{1}{\mu_0} \vec{B} - \vec{M}) = 0
$$

Since $\vec{\nabla} \cdot \vec{B} = 0$ everywhere, then $\vec{B}$ can be described as the curl of a vector potential field, $\vec{A}$. 

$$
\therefore \vec{\nabla} \times (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A}) - \vec{M}) = 0
$$

The cross product is distributive over vector addition, so:

$$
\vec{\nabla} \times (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A})) = \vec{\nabla} \times \vec{M}
$$

To find the weak form, we then multiply both sides by some arbitrary test function $v$ in the function space $V$:

$$
(\vec{\nabla} \times (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A}))) \cdot v = (\vec{\nabla} \times \vec{M}) \cdot v
$$

Then integrate over the collimator space $\Omega$:

$$
\int_{\Omega} (\vec{\nabla} \times (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A}))) \cdot v dA = \int_{\Omega} (\vec{\nabla} \times \vec{M}) \cdot v dA
$$

We can rearrange the left hand side using this proerty of the divergence and curl:

$$
\mathbf{v}\cdot(\nabla\times\mathbf{F}) = \mathbf{F}\cdot(\nabla\times\mathbf{v}) + \nabla\cdot(\mathbf{F}\times\mathbf{v})
$$

Leffing $(\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A}))$ be $\vec{F}$, the left side of the equation is therefore:

$$
\int_\Omega \mathbf{v}\cdot(\nabla\times\mathbf{F})\,dx = \int_\Omega \mathbf{F}\cdot(\nabla\times\mathbf{v})\,dx + \int_\Omega \nabla\cdot(\mathbf{F}\times\mathbf{v})\,dx
$$

The second component here is the divergence of a field integrated over a volume, so can be changed to a surface integral using the divergence theorem:

$$
\int_\Omega (\nabla\times\mathbf{F})\cdot\mathbf{v}\,dx = \int_\Omega \mathbf{F}\cdot(\nabla\times\mathbf{v})\,dx + \oint_{\partial\Omega}(\mathbf{F}\times\mathbf{v})\cdot\mathbf{n}\,ds
$$

If we impose the condition that $\vec{A} = 0$ on the boundary of the problem, then the left hand side becomes:

$$
\int_\Omega \mathbf{F}\cdot(\nabla\times\mathbf{v})\,dx
$$

Doing the same to the right hand side, and setting the magnetisation field to zero outside of the magnets, we then have:

$$
\int_\Omega \mathbf{M}\cdot(\nabla\times\mathbf{v})\,dx - \oint_{\partial\Omega}(\mathbf{M}\times\mathbf{v})\cdot\mathbf{n}\,ds
$$

$$
=\int_\Omega \mathbf{M}\cdot(\nabla\times\mathbf{v})\,dx
$$

$$
\therefore \int_\Omega \mathbf{F}\cdot(\nabla\times\mathbf{v})\,dx = \int_\Omega \mathbf{M}\cdot(\nabla\times\mathbf{v})\,dx
$$


In [ ]:
# Define a function space for the magnetic scalar potential
V = dolfinx.fem.functionspace(mesh, ("N1curl", 1)) # Nedelec space of degree 1 - often used for Maxwell's eqns

# Discontinuous scalar space for piecewise-constant material data
Q = dolfinx.fem.functionspace(mesh, ("DG", 0)) 
mu = dolfinx.fem.Function(Q)

magnet_cells = cell_tags.find(physicalgroups['magnets']['tag'])
air_cells = cell_tags.find(physicalgroups['collimator']['tag'])

mu.x.array[magnet_cells] = variables.mu0 * variables.mu_r_magnet # Define magnetic permeability for the magnets and vacuum
mu.x.array[air_cells] = variables.mu0

W = dolfinx.fem.functionspace(mesh, ("DG", 0, (3,))) # Discontinuous Galerkin function space with order 0 and ? dimensions 3,?
M = dolfinx.fem.Function(W) # Function M, magnetisation vector?

Mmag = variables.Br / variables.mu0 # A/m (B_total = mu_0 (H + M), therefore M = B_r / mu_0 in absence of H field)

# TODO do this again with other polarity of magnets. May need to split them into two physical groups. 

axis = np.array([0.0, 0.0, 1.0])  # magnetization direction, e.g. bar's long axis

M_dofs = dolfinx.fem.locate_dofs_topological(W, mesh.topology.dim, magnet_cells)
# set only cells in the magnet; simplest robust way is to loop over blocks
M_array = M.x.array.reshape(-1, 3)
M_array[magnet_cells, :] = Mmag * axis
M.x.array[:] = M_array.flatten()

# So we have three function spaces; V (with A & v), for the magnetic vector potential, Q (with mu) for the magnetic permeability, and W (with M) for the magnetisation

(289454, 3)


In [15]:
A = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

a = (1 / mu) * ufl.inner(ufl.curl(A), ufl.curl(v)) * ufl.dx
L = ufl.inner(M, ufl.curl(v)) * ufl.dx   # zero outside magnet automatically

# Zero tangential A on the outer boundary (far-field truncation)
# TODO I will probably have to build another entire cylinder around the whole thing where the field can go to zero. Noooooooo </3
# UP TO HERE
OUTER_TAG = physicalgroups['collimator_surface']['tag']
outer_facets = facet_tags.find(OUTER_TAG)
outer_dofs = dolfinx.fem.locate_dofs_topological(V, mesh.topology.dim - 1, outer_facets)
bc = dolfinx.fem.dirichletbc(dolfinx.fem.Function(V), outer_dofs)  # A = 0 there

problem = dolfinx.fem.petsc.LinearProblem(a, L, bcs=[bc],
                         petsc_options={"ksp_type": "cg", "pc_type": "hypre",
                                         "pc_hypre_type": "ams"})
A_h = problem.solve()

Error: error code 83
[0] KSPSolve() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/ksp/interface/itfunc.c:1079
[0] KSPSolve_Private() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/ksp/interface/itfunc.c:832
[0] KSPSetUp() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/ksp/interface/itfunc.c:415
[0] PCSetUp() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/pc/interface/precon.c:1080
[0] PCSetUp_HYPRE() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/pc/impls/hypre/hypre.c:295
[0] HYPRE AMS preconditioner needs either the coordinate vectors via PCSetCoordinates() or the edge constant vectors via PCHYPRESetEdgeConstantVectors() or the interpolation matrix via PCHYPRESetInterpolations()

# Define boundary conditions

There are certain boundary conditions that apply to the mesh, which are implemented before simulation can begin.

The majority of these are Dirichlet boundary conditions, which take the form:

$$ \phi(\vec{r}) |_{s} = f(\vec{r}) $$

Where $f(\vec{r})$ is some known function, and $\phi(\vec{r}) |_{s}$ is the electric potential evaluated at the surface S. For example, the cathode, anode, and trigger are held at a known potential difference.

Typically, electromagnetic problems set the potential to zero at an infinite distance from the charges. However, this is not implemented here for multiple reasons:

1)  FEniCSx is not able to model infinite distances. 
    When modelling infinite values computationally, the typical strategy is to choose a suitably large number instead of infinity (i.e., set the potential to zero at a distance of 10 metres from the thruster and collimator.). However, the size of the mesh must be very small to accurately capture ion-ion interactions within the plasma. Therefore it would require significant computing power to model a sufficiently large space. 

2)  The potential at an infinite distance is unlikely to be zero in real life, due to the influence of the Earth's magnetic field (during both vacuum chamber testing and in low-Earth orbit) as well as the ionosphere, other satellites, and solar weather in orbit.

Instead, a background magnetic field is modelled. Since the simulation will be experimentally verified in a vacuum chamber on Earth, the background magnetic field will be approximately 1 Gauss. In low-Earth orbit, this value would be lower. 

To define the boundary conditions, a function space $V$ is created on the imported mesh. 


In [6]:
# This process is described in the FEniCSx tutorial documentation here: https://jsdokken.com/dolfinx-tutorial/chapter3/subdomains.html#convert-msh-files-to-xdmf-using-meshio

# Define a function space on the mesh.
# Note: 
# - Lagrangian is a smooth function L described here: https://defelement.org/elements/lagrange.html
# - Degree 1 means points are defined at the vertices, whereas (for example) degree 2 has points at the midpoints as well. 

V = dolfinx.fem.functionspace(mesh, ("Lagrange", 1))

# Get the dimension of the entity to apply the BC on (Dirichlet conditions are applied on the boundary, 
# so will have dimension 1 lower than the problem dimension. The boundary is a 2d surface in this case.)
fdim = mesh.topology.dim - 1


# Locate degrees-of-freedom belonging to mesh entities topologically
cathode_dofs = dolfinx.fem.locate_dofs_topological(V, fdim, facet_tags.find(physicalgroups['cathode_surface']['tag']))
anode_dofs = dolfinx.fem.locate_dofs_topological(V, fdim, facet_tags.find(physicalgroups['anode_surface']['tag']))
magnet_dofs = dolfinx.fem.locate_dofs_topological(V, fdim, facet_tags.find(physicalgroups['magnets_surface']['tag']))

# Apply a dirichlet BC for the different values
bcs = [#dolfinx.fem.dirichletbc(dolfinx.default_scalar_type(variables.V_c), cathode_dofs, V), 
       #dolfinx.fem.dirichletbc(dolfinx.default_scalar_type(variables.V_a), anode_dofs, V), 
       #dolfinx.fem.dirichletbc(dolfinx.default_scalar_type(variables.V_mag), magnet_dofs, V)
       ]

# TODO I think the anode and cathode have an electric scalar potential (volts) not a magnetic scalar potential (Amps)

# NOT SURE about after this:
import ufl 
import dolfinx.fem.petsc
# H(curl) Nédélec space for A
degree = 1
V = dolfinx.fem.functionspace(mesh, ("N1curl", degree))
mu0 = 4 * np.pi * 1e-7

# Discontinuous scalar space for piecewise-constant material data
Q = dolfinx.fem.functionspace(mesh, ("DG", 0))
mu = dolfinx.fem.Function(Q)

MAGNET_TAG = physicalgroups['magnets']['tag']   # your gmsh physical group ids
AIR_TAG = physicalgroups['collimator']['tag']

magnet_cells = cell_tags.find(MAGNET_TAG)
air_cells = cell_tags.find(AIR_TAG)

mu_r_magnet = 1.05  # typical for NdFeB; ~1.0 is fine as an approximation
mu.x.array[magnet_cells] = mu0 * mu_r_magnet
mu.x.array[air_cells] = mu0

W = dolfinx.fem.functionspace(mesh, ("DG", 0, (3,)))
M = dolfinx.fem.Function(W)

Br = 1.2          # remanence in Tesla, e.g. N42 NdFeB
Mmag = Br / mu0    # A/m
axis = np.array([0.0, 0.0, 1.0])  # magnetization direction, e.g. bar's long axis

M_dofs = dolfinx.fem.locate_dofs_topological(W, mesh.topology.dim, magnet_cells)
# set only cells in the magnet; simplest robust way is to loop over blocks
M_array = M.x.array.reshape(-1, 3)
M_array[magnet_cells, :] = Mmag * axis
M.x.array[:] = M_array.flatten()

A = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

a = (1 / mu) * ufl.inner(ufl.curl(A), ufl.curl(v)) * ufl.dx
L = ufl.inner(M, ufl.curl(v)) * ufl.dx   # zero outside magnet automatically

# Zero tangential A on the outer boundary (far-field truncation)
OUTER_TAG = physicalgroups['collimator_surface']['tag']
outer_facets = facet_tags.find(OUTER_TAG)
outer_dofs = dolfinx.fem.locate_dofs_topological(V, mesh.topology.dim - 1, outer_facets)
bc = dolfinx.fem.dirichletbc(dolfinx.fem.Function(V), outer_dofs)  # A = 0 there

problem = dolfinx.fem.petsc.LinearProblem(a, L, bcs=[bc],
                         petsc_options={"ksp_type": "cg", "pc_type": "hypre",
                                         "pc_hypre_type": "ams"})
A_h = problem.solve()

W_dg = dolfinx.fem.functionspace(mesh, ("DG", degree - 1, (3,)))
B = dolfinx.fem.Function(W_dg)
B_expr = dolfinx.fem.Expression(ufl.curl(A_h), W_dg.element.interpolation_points())
B.interpolate(B_expr)

Error: error code 83
[0] KSPSolve() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/ksp/interface/itfunc.c:1079
[0] KSPSolve_Private() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/ksp/interface/itfunc.c:832
[0] KSPSetUp() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/ksp/interface/itfunc.c:415
[0] PCSetUp() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/pc/interface/precon.c:1080
[0] PCSetUp_HYPRE() at /home/conda/feedstock_root/build_artifacts/petsc_1711823677387/work/src/ksp/pc/impls/hypre/hypre.c:295
[0] HYPRE AMS preconditioner needs either the coordinate vectors via PCSetCoordinates() or the edge constant vectors via PCHYPRESetEdgeConstantVectors() or the interpolation matrix via PCHYPRESetInterpolations()

# Define Particles

Once the boundary conditions have been defined, then particles are distributed randomly within the thruster. 

They are assigned an initial position and velocity.

In [ ]:
xp = np.ones((variables.n_p, 3)) # Particle position vector

# Position all particles at the trigger
xp[:, 0] = variables.l_trig
xp[:, 1] = 0
xp[:, 2] = 0

vp = np.zeros(xp.shape) # Particle velocity vector
psip = np.zeros(xp.shape[0]) # Particle property scalar. This must be constant. 


# Fill every cell with 25 particles. 
# xp is the position vector of every particle, and p2cell maps their location to which cell they belong to. 
xp, p2cell = pyleopart.mesh_fill(mesh._cpp_object, 25)
print(f"num paticles: {xp.shape[0]}")
ptcls = pyleopart.Particles(xp, p2cell)

# Moving the particles

The force $\vec{F}$ due to magnetic and electric fields $\vec{B}$ and $\vec{E}$ acting on a charge q with velocity $\vec{v}$ is given by:

$$
\vec{F} = q(\vec{E} +(\vec{v} \times \vec{B}))
$$

Therefore their velocity at time dt is given by:

$$
m \frac{d\vec{v}}{dt} = q(\vec{E} +(\vec{v} \times \vec{B}))
$$

$$
\therefore \frac{d\vec{v}}{dt} = \frac{q}{m}(\vec{E} +(\vec{v} \times \vec{B}))
$$

Where dt is so small that we assume that $\vec{E}$ and $\vec{B}$ are unchanged over dt, and the movement of the particle over dt is treated as linear. 

The position of the particle after time t is given as:

$$ 
\frac{d\vec{x}}{dt} = \vec{v}
$$

These coupled linear PDEs can then be solved together. 

LeoPart requires three variables that characterise the particles:

$$
\frac{d\vec{x}}{dt} = a_h(\vec{x}, t) = \vec{v}
$$

Where $a_h(\vec{x}, t)$ is a vector-valued velocity field;

$$ 
\frac{d\psi}{dt} = 0
$$
Where $\psi$ is some constant that characterises the particles. In our case, this will be $\frac{q_{ion}}{m_{ion}}$, which is the charge and mass of the molybdenum ions. 

and 
$$
\frac{d\vec{v}}{dt} = 0
$$

In [ ]:
tableau = pyleopart.tableaus.order4.classic() # Choose the tableau (matrix of coefficients) to use for the Runge-Kutta method
tableau.check_and_create_fields(ptcls)

charge_mass_space = dolfinx.fem.FunctionSpace(mesh, ("DG", 0))
Q = dolfinx.fem.Function(charge_mass_space)
Q.interpolate(lambda x: np.full(x.shape[1], variables.m_ion, dtype=np.float64))
Q.x.scatter_forward()

# Add the Q function space to the particles:

charge_mass_label = "Q"
ptcls.add_field(charge_mass_label, [1])
pyleopart.transfer_to_particles(ptcls, ptcls.field(charge_mass_label), Q._cpp_object)

# Add the electric field:
e_space = dolfinx.fem.functionspace(mesh, ("DG", 2))
E = dolfinx.fem.Function(e_space)
E.interpolate()